##### CP201A Lecture: The Power of Python

You already met Jupyter and the Python basics in Lab 1, so today we skip ahead and start at the end. The goal is to see what Python can do for a real planning question before we build the pieces. We'll ask: **does a neighborhood's 1930s redlining grade still shape conditions there today?**

Run each cell with **Shift + Enter**. If a cell throws a red error, that's normal. Watch mine and catch the next one.

In [ ]:
# One-time setup. These mapping packages may already be on Datahub; this only installs what is missing.
import importlib, subprocess, sys
for pkg in ["folium", "geopandas", "mapclassify"]:
    if importlib.util.find_spec(pkg) is None:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg])

In [ ]:
# A quick warm-up so we know the kernel is running
print("Let's see what Python can do.")

## 1. Example of the Power of Programming

We're going to see whether a neighborhood's redlining grade correlates with conditions in a census tract today, using data from the ACS. We'll introduce a couple of ideas, like correlation, that we come back to formally later.

First we load some libraries. We'll learn what libraries are in the coming weeks. For now, just run the cell.

In [ ]:
# Pre-written Python packages we'll use to read, clean, analyze, and map the data.
import os
import json
import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import folium
import geopandas as gpd
import seaborn as sns
plt.style.use('fivethirtyeight')
%matplotlib inline
pd.options.display.float_format = '{:.2f}'.format

### 1.1 Redlining Map for Oakland

Let's bring in a 1937 redlining (HOLC) map for Oakland. Python reads geojson files easily.

In [ ]:
# Import the map file for Oakland
geo_json_data = json.load(open('CAOakland1937.geojson'))

In [ ]:
# Assign each HOLC grade a color
def my_color_function(feature):
    if feature['properties']['holc_grade'] == 'A':
        return '#98ff98'
    elif feature['properties']['holc_grade'] == 'B':
        return '#5bc0de'
    elif feature['properties']['holc_grade'] == 'C':
        return '#ffe200'
    else:
        return '#ff00aa'

In [ ]:
# Draw the map, centered on Oakland
m = folium.Map([37.8044, -122.271], tiles='cartodbpositron', zoom_start=12)
folium.GeoJson(
    geo_json_data,
    style_function=lambda feature: {
        'fillColor': my_color_function(feature),
        'color': 'black',
        'weight': 1,
    }
).add_to(m)
m

### 1.2 Comparing Redlining Scores with Conditions Today

Each HOLC grade has a number. Grade 1 (green) neighborhoods were considered "safe" to lend in. Grade 2 (blue) were also "safe" and could get FHA loans. Grade 3 (yellow) could still get loans, but appraisers flagged them as declining. Grade 4 (red) were called "hazardous," and the federal government refused to guarantee mortgages there.

In [ ]:
# Read in census and ACS data for Oakland tracts (1990 and 2017)
df_2017 = pd.read_excel('Oakland_1990_2017_Data.xlsx', dtype={"Census Tract": str})

In [ ]:
# Take a look at the data
df_2017.head()

### 1.3 Correlation

A correlation statistic tells us the direction and strength of the relationship between two variables. The tool below lets us compare the redlining grade against different tract characteristics in 1990 and 2017. Run the next two cells, then change the dropdowns.

In [ ]:
# Widget tools
import ipywidgets as widgets
from ipywidgets import interact, interactive, fixed, interact_manual
from IPython.display import display

In [ ]:
# Pick any two columns to compare. Opens on redlining grade vs. the 2017 poverty rate.
a_dd = widgets.Dropdown(options=df_2017.columns.tolist(), description='X')
b_dd = widgets.Dropdown(options=df_2017.columns.tolist(), description='Y')
out = widgets.Output()

def update(_=None):
    with out:
        out.clear_output(wait=True)
        x = pd.to_numeric(df_2017[a_dd.value], errors='coerce')
        y = pd.to_numeric(df_2017[b_dd.value], errors='coerce')
        ok = x.notna() & y.notna()
        x, y = x[ok].to_numpy(), y[ok].to_numpy()
        fig, ax = plt.subplots(figsize=(12, 6))
        ax.scatter(x, y, alpha=0.5, label='Census tracts')
        if len(x) > 1 and np.ptp(x) > 0:
            m_, c_ = np.polyfit(x, y, 1)
            X = np.linspace(x.min(), x.max(), 200)
            ax.plot(X, m_ * X + c_, linewidth=2, label='Regression line')
            r = np.corrcoef(x, y)[0, 1]
            ax.set_title(f'{a_dd.value} vs {b_dd.value}   |   r = {r:.3f}')
        else:
            ax.set_title(f'{a_dd.value} vs {b_dd.value}')
        ax.set_xlabel(a_dd.value)
        ax.set_ylabel(b_dd.value)
        ax.legend()
        plt.show()

a_dd.observe(update, names='value')
b_dd.observe(update, names='value')

# Sensible defaults so the demo opens on the key relationship
a_dd.value = 'Redlining Grade'
b_dd.value = 'Poverty Rate 2017'

display(widgets.HBox([a_dd, b_dd]), out)
update()

### 1.4 Reading the results

Put **Redlining Grade** in X and **Poverty Rate 2017** in Y. The correlation is about **+0.41** and the line slopes up: as a tract's redlining grade gets worse (4 = redlined), the poverty rate rises, even 60 years later.

Now try **Redlining Grade** against **Percent with a BA Degree or Higher 2017**. This time the correlation is about **-0.42**: tracts that were redlined have fewer residents with a BA today. (Anything past about 0.3 in size is a meaningful relationship. We'll test significance properly later.)

### 1.5 Map the Indicators

Now that we're in Python, it's easy to map any indicator in the dataset. The data also includes a gentrification "risk" score based on the Urban Displacement Project maps: higher means higher risk.

In [ ]:
# Join the tract shapes to the data
tracts_gdf = gpd.read_file('Oakland_Tracts.shp')
merged_gdf = tracts_gdf.set_index("GEOID_2").join(df_2017.set_index("Census Tract"))

In [ ]:
# List the variables you can map
df_2017.columns

In [ ]:
# Map an indicator. Change the column name and title to map something else.
figure, ax = plt.subplots(figsize=(14, 10))
ax = merged_gdf.plot(column="Median House Value 1990", legend=True, ax=ax, cmap="Blues")
lims = plt.axis("equal")
ax.set_axis_off()
ax.set_title('Median House Value 1990', fontdict={'fontsize': 25})
plt.show()

## Fascinating, right? And powerful once we put it to work on planning questions.

### Resetting this notebook

To run through it again on your own: click the bCourses link, then from the **Kernel** menu choose **Restart & Clear Output**. That gives you a fresh copy. Just remember to run the cells in order.